In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# MODELS
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, recall_score, roc_auc_score, precision_score, accuracy_score, f1_score

In [6]:
df = pd.read_csv("merged_data.csv")
df['Diabetes_Status'] = df['Diabetes_Status'].map({1:1, 2:0})
df = df.drop(columns=['SEQN'])
# df.hist(figsize=(20,20))
# from tabulate import tabulate

# nulls = df.isnull().sum().reset_index()
# nulls.columns = ['Column', 'Missing Values']
# print(tabulate(nulls, headers='keys', tablefmt='psql'))
# print(df.shape)

#CLEANING
df_clean = df.dropna()
df_clean = df_clean[(df_clean["Diabetes_Status"] != 3) & (df_clean["Diabetes_Status"] != 9)]
df_clean.shape
# df_clean.hist(figsize=(20,20))
# print(df_clean.isnull().sum())

X_clinical = df_clean.drop(columns=["Diabetes_Status"])
y_clinical = df_clean["Diabetes_Status"]

X_train_clinical, X_test_clinical, y_train_clinical, y_test_clinical = train_test_split(
    X_clinical, y_clinical, test_size=0.2, stratify=y_clinical, random_state=42
)

In [7]:
numeric_features = X_train_clinical.columns.tolist()

over = SMOTE()
under = RandomUnderSampler()

preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), numeric_features)],
    remainder='passthrough'  # keep any non-numeric features if present
)

preprocessor_SMOTE = ColumnTransformer(
    transformers=[('num', StandardScaler(), numeric_features)],
    remainder='passthrough'  # keep any non-numeric features if present
)

In [8]:
best_pipe_clinical = ImbPipeline([
    ('preprocess', preprocessor),
    ('over', over),
    ('under', under),
    ('model', LogisticRegression(
        max_iter=1000,
        random_state=42,
        penalty='l2',
        C=0.1,
        class_weight='balanced'
    ))
])

best_pipe_clinical.fit(X_train_clinical, y_train_clinical)
clinical_preds_train = best_pipe_clinical.predict_proba(X_train_clinical)[:, 1]

# LIFESTYLE

In [9]:
df_nondiab = pd.read_csv("nondiabetic_merged_filled.csv")
df_nondiab["diabetes_status"] = 0
df_nondiab.drop(columns=["Respondent_ID"], inplace=True)
df_nondiab.rename(columns={"Sex_num": "Gender"}, inplace=True)
df_nondiab.info()

df_diab = pd.read_csv("diab_merge_final.csv")
df_diab["diabetes_status"] = 1

merged_df = pd.concat([df_nondiab, df_diab], ignore_index=True)

merged_df.drop(columns=["BMI", "child_diab_ord", "Course", "Year Level", "Height_cm", "Weight_kg", "Age", "Gender", "alcohol_ord.1", "concern_level_ord"], inplace=True)
merged_df.fillna(merged_df.median(numeric_only=True), inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 378 entries, 0 to 377
Data columns (total 33 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Gender                 378 non-null    int64  
 1   alcohol_ord            378 non-null    int64  
 2   Age                    378 non-null    int64  
 3   Height_cm              377 non-null    float64
 4   Weight_kg              377 non-null    float64
 5   BMI                    376 non-null    float64
 6   Course                 355 non-null    object 
 7   Year Level             355 non-null    object 
 8   fruit_freq_ord         378 non-null    int64  
 9   veg_freq_ord           378 non-null    int64  
 10  sweets_freq_ord        378 non-null    int64  
 11  fastfood_freq_ord      378 non-null    int64  
 12  processed_freq_ord     378 non-null    int64  
 13  sweetdrink_freq_ord    378 non-null    int64  
 14  fried_food_freq_ord    378 non-null    int64  
 15  lose_w

In [10]:
numeric_features = [ 'sweets_freq_ord', 'fruit_freq_ord', 'veg_freq_ord', 'fastfood_freq_ord',
    'processed_freq_ord', 'sweetdrink_freq_ord', 'fried_food_freq_ord',
    'lose_weight_ord', 'exercise_yes_ord', 'exercise_freq_ord', 'exercise_duration_ord'
,'sedentary_hours_ord',"activity_level_ord", "transpo_ord", "sleep_ord", "alcohol_ord", "father_diab_ord", "mother_diab_ord", "sister_diab_ord", "brother_diab_ord","extended_diab_ord"]

over = SMOTE()
under = RandomUnderSampler()

preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), numeric_features)],
    remainder='passthrough'  # keep any non-numeric features if present
)


In [11]:
X_lifestyle = merged_df.drop(columns=["diabetes_status"])
y_lifestyle = merged_df["diabetes_status"]

X_train_lifestyle, X_test_lifestyle, y_train_lifestyle, y_test_lifestyle = train_test_split(
    X_lifestyle, y_lifestyle, test_size=0.2, stratify=y_lifestyle, random_state=42
)

In [12]:
best_pipe_lifestyle = ImbPipeline([
    ('preprocess', preprocessor),
    ('over', over),
    ('model', RandomForestClassifier(random_state=42))
])
best_pipe_lifestyle.fit(X_train_lifestyle, y_train_lifestyle)
lifestyle_preds_train = best_pipe_lifestyle.predict_proba(X_train_lifestyle)[:, 1]



In [13]:
meta_features_train = np.column_stack([clinical_preds_train, lifestyle_preds_train])
meta_model = LogisticRegression()
meta_model.fit(meta_features_train, y_train_clinical)

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 2533 and the array at index 1 has size 356